# DISCRIMINANT ANALYSIS

In this coding assignment you are to implement a Minimum Risk Bayes Decision Theoretic classifier. Use the training set to train the classifier and the validation set to evaluate the accuracy.

Assume the following:
1. All conditional density functions are multivariate Gaussian
2. Each class has its own covariance matrix
3. Equal prior probabilities
4. 0-1 loss function

You may use *off-the-shell* functions to compute the mean and the covariance, but obviously you are not
allowed to use the actual algorithm such as *sklearn.discriminant_analysis*.

Print the actual and predicted class labels.
Print the average classification accuracy. Average accuracy should be at least 90%.

  y=1, y_hat=1

  y=3, y_hat=3

  y=1, y_hat=1

  y=2, y_hat=2

  y=2, y_hat=2

  y=1, y_hat=1

  y=2, y_hat=2

  y=1, y_hat=1

  y=2, y_hat=3

  y=3, y_hat=3

  y=3, y_hat=3

  y=3, y_hat=3

  y=2, y_hat=2

  y=3, y_hat=3
  
  y=1, y_hat=1

Average classification accuracy = 0.9333

## Load datasets

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Load training data - 135 observations, 4 features, 3 classes,
df = pd.read_csv("iris_corrupted_training_dataset.csv")
print(df.head())
df = df.values
train_data = df

# Load validation data - 15 samples
df = pd.read_csv("iris_validation_dataset.csv")
print(df.head())
df = df.values
val_data = df

   sepal_length   sepal_width   petal_length   petal_width   class
0        5.7147        2.6743         3.2696       1.65440       2
1        5.1734        3.7374         5.9442       3.00050       3
2        7.3776        3.1505         3.3543       0.64839       2
3        6.4908        2.3983         3.3917       1.54950       2
4        6.8182        3.4016         4.7495       0.57970       3
   sepal_length   sepal_width   petal_length   petal_width   class
0           4.4           2.9            1.4           0.2       1
1           6.7           3.0            5.2           2.3       3
2           4.9           3.1            1.5           0.2       1
3           5.1           2.5            3.0           1.1       2
4           6.1           3.0            4.6           1.4       2


In [2]:
## Your code goes here ...

X_train = train_data[:, :-1]
y_train = train_data[:, -1]

X_val = val_data[:, :-1]
y_val = val_data[:, -1]

print("Shape of X_train:", X_train.shape)
print("Shape of y_train:", y_train.shape)
print("Shape of X_val:", X_val.shape)
print("Shape of y_val:", y_val.shape)

Shape of X_train: (135, 4)
Shape of y_train: (135,)
Shape of X_val: (15, 4)
Shape of y_val: (15,)


In [5]:
unique_classes = np.unique(y_train)

mu_mle = {}
sigma_mle = {}

for class_label in unique_classes:
    # Filter training data for the current class
    X_class = X_train[y_train == class_label]
    # Calculate mean vector (mu_MLE_i)
    mu_mle[class_label] = np.mean(X_class, axis=0)
    # Calculate covariance matrix (Sigma_MLE_i)
    sigma_mle[class_label] = np.cov(X_class, rowvar=False)
    print(f"Class {int(class_label)} Mean (μ_MLE_{int(class_label)}):\n{mu_mle[class_label]}\n")
    print(f"Class {int(class_label)} Covariance (Σ_MLE_{int(class_label)}):\n{sigma_mle[class_label]}\n")

Class 1 Mean (μ_MLE_1):
[4.80081778 3.48799556 1.26920989 0.34787733]

Class 1 Covariance (Σ_MLE_1):
[[ 0.73847372 -0.09788292  0.162097    0.09430334]
 [-0.09788292  1.04517177  0.08250472  0.06122466]
 [ 0.162097    0.08250472  0.75386746  0.07747734]
 [ 0.09430334  0.06122466  0.07747734  0.51347455]]

Class 2 Mean (μ_MLE_2):
[6.06588222 2.82287978 4.26241333 1.10785197]

Class 2 Covariance (Σ_MLE_2):
[[ 1.02666705  0.16051089  0.28736137 -0.10850815]
 [ 0.16051089  0.80414317  0.20221368 -0.07318826]
 [ 0.28736137  0.20221368  0.74048204 -0.04380217]
 [-0.10850815 -0.07318826 -0.04380217  0.69674064]]

Class 3 Mean (μ_MLE_3):
[6.42966    2.95656956 5.55874667 1.92476547]

Class 3 Covariance (Σ_MLE_3):
[[1.36272732 0.26608677 0.44568822 0.30336696]
 [0.26608677 1.03934606 0.12853287 0.18437967]
 [0.44568822 0.12853287 0.69605886 0.23021863]
 [0.30336696 0.18437967 0.23021863 0.85756954]]



In [7]:
def discriminant_function(x, mu_i, sigma_i, d, prior_prob_i):
    # Ensure sigma_i is positive definite to avoid issues with inverse and determinant
    det_sigma_i = np.linalg.det(sigma_i)
    if det_sigma_i <= 0:
        # Regularize covariance matrix: add a small identity matrix
        sigma_i = sigma_i + np.eye(d) * 1e-6
        det_sigma_i = np.linalg.det(sigma_i)
        # If still non-positive after regularization, return a very low score
        if det_sigma_i <= 0:
            return -np.inf
    inv_sigma_i = np.linalg.inv(sigma_i)
    x_minus_mu_i = x - mu_i
    # Mahalanobis distance term: (x - mu)^T * Sigma^-1 * (x - mu)
    maha_dist_term = np.dot(np.dot(x_minus_mu_i.T, inv_sigma_i), x_minus_mu_i)
    # Standard discriminant function for a Bayes classifier with multivariate Gaussian densities
    # g_i(x) = -0.5 * (x - μ_i)^T Σ_i^-1 (x - μ_i) - 0.5 * ln(|Σ_i|) - 0.5 * d * ln(2π) + ln(P(ω_i))
    g_i_x = -0.5 * maha_dist_term - 0.5 * np.log(det_sigma_i) - 0.5 * d * np.log(2 * np.pi) + np.log(prior_prob_i)
    return g_i_x

# Get number of features
d = X_train.shape[1]
# Calculate equal prior probabilities
prior_prob_per_class = 1.0 / len(unique_classes)
# Store predicted labels
y_pred = []
actual_predicted_labels = []

# Iterate over each validation data point
for i in range(len(X_val)):
    x_val_point = X_val[i]
    actual_label = y_val[i]
    discriminant_scores = {}
    # Calculate discriminant function for each class
    for class_label in unique_classes:
        mu = mu_mle[class_label]
        sigma = sigma_mle[class_label]
        score = discriminant_function(x_val_point, mu, sigma, d, prior_prob_per_class)
        discriminant_scores[class_label] = score
    # Find the class with the maximum discriminant score
    predicted_label = max(discriminant_scores, key=discriminant_scores.get)
    y_pred.append(predicted_label)
    actual_predicted_labels.append(f"y={int(actual_label)}, y_hat={int(predicted_label)}")

# Print actual and predicted class labels
for pair in actual_predicted_labels:
    print(pair)
# Calculate accuracy
correct_predictions = np.sum(np.array(y_pred) == y_val)
average_accuracy = correct_predictions / len(y_val)
print(f"\nAverage classification accuracy = {average_accuracy:.4f}")

y=1, y_hat=1
y=3, y_hat=3
y=1, y_hat=1
y=2, y_hat=2
y=2, y_hat=2
y=1, y_hat=1
y=2, y_hat=2
y=1, y_hat=1
y=2, y_hat=3
y=3, y_hat=3
y=3, y_hat=3
y=3, y_hat=3
y=2, y_hat=2
y=3, y_hat=3
y=1, y_hat=1

Average classification accuracy = 0.9333
